In [1]:
import os
import random
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)


In [3]:
SEED = 42

def fix_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

fix_seeds()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")


Running on: cuda


## 1. Data Loading & Exploration

In [4]:
ROOT = "/kaggle/input/datasets/subhajournal/busi-breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"

paths, labels = [], []

for dirpath, _, filenames in os.walk(ROOT):
    for fname in filenames:
        if not fname.lower().endswith(".png") or "mask" in fname.lower():
            continue

        folder = dirpath.lower()
        if "benign" in folder:
            cls = "benign"
        elif "malignant" in folder:
            cls = "malignant"
        elif "normal" in folder:
            cls = "normal"
        else:
            continue

        paths.append(os.path.join(dirpath, fname))
        labels.append(cls)

data_df = pd.DataFrame({"filepath": paths, "class": labels})

print(f"Total scans loaded : {len(data_df)}")
print("\nClass distribution:")
print(data_df["class"].value_counts())


Total scans loaded : 780

Class distribution:
class
benign       437
malignant    210
normal       133
Name: count, dtype: int64


In [5]:
CLASS_MAP = {"normal": 0, "benign": 1, "malignant": 2}
data_df["label"] = data_df["class"].map(CLASS_MAP)

train_val_df, test_df = train_test_split(
    data_df, test_size=0.15, stratify=data_df["label"], random_state=SEED
)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.15, stratify=train_val_df["label"], random_state=SEED
)

print(f"Train : {len(train_df)} samples")
print(f"Val   : {len(val_df)} samples")
print(f"Test  : {len(test_df)} samples")

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())


Train : 563 samples
Val   : 100 samples
Test  : 117 samples

Train label distribution:
label
0     96
1    315
2    152
Name: count, dtype: int64


## 2. Dataset & Transforms

In [6]:
class BUSIDataset(Dataset):
    
    def __init__(self, dataframe, transform=None):
        self.records = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        target = torch.tensor(row["label"], dtype=torch.long)
        return img, target


In [7]:
IMG_SIZE = 224
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD  = [0.5, 0.5, 0.5]

standard_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])

augmented_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=12),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ToTensor(),
    T.Normalize(NORM_MEAN, NORM_STD),
])


In [8]:
BATCH = 32

ds_train     = BUSIDataset(train_df, standard_tfm)
ds_train_aug = BUSIDataset(train_df, augmented_tfm)
ds_val       = BUSIDataset(val_df,   standard_tfm)
ds_test      = BUSIDataset(test_df,  standard_tfm)

loader_kwargs = dict(num_workers=2, pin_memory=(DEVICE.type == "cuda"))

train_loader     = DataLoader(ds_train,     batch_size=BATCH, shuffle=True,  **loader_kwargs)
val_loader       = DataLoader(ds_val,       batch_size=BATCH, shuffle=False, **loader_kwargs)
test_loader      = DataLoader(ds_test,      batch_size=BATCH, shuffle=False, **loader_kwargs)
train_loader_aug = DataLoader(ds_train_aug, batch_size=BATCH, shuffle=True,  **loader_kwargs)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")


Train batches : 18
Val batches   : 4
Test batches  : 4


### 2.1 Weighted Oversampling Loader (for class imbalance)

In [9]:
counts = train_df["label"].value_counts().sort_index()
print("Train class counts:\n", counts)

inv_weights = 1.0 / counts
sample_w = train_df["label"].map(inv_weights).values

weighted_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_w),
    num_samples=len(sample_w),
    replacement=True
)

train_loader_over = DataLoader(
    ds_train,
    batch_size=BATCH,
    sampler=weighted_sampler,
    **loader_kwargs
)

print(f"Oversampled loader batches: {len(train_loader_over)}")


Train class counts:
 label
0     96
1    315
2    152
Name: count, dtype: int64
Oversampled loader batches: 18


## 3. Model Architectures

In [10]:

class StdConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, stride=1, pad=1):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel, stride, pad, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layer(x)


class DSConvBlock(nn.Module):
  

    def __init__(self, in_channels, out_channels, kernel=3, stride=1, pad=1):
        super().__init__()
        
        self.dw_conv = nn.Conv2d(
            in_channels, in_channels, kernel, stride, pad,
            groups=in_channels, bias=False
        )
        self.dw_bn  = nn.BatchNorm2d(in_channels)

        self.pw_conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.pw_bn  = nn.BatchNorm2d(out_channels)

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act(self.dw_bn(self.dw_conv(x)))
        x = self.act(self.pw_bn(self.pw_conv(x)))
        return x


In [11]:
class PlainCNN(nn.Module):
  
    def __init__(self, num_classes=3):
        super().__init__()
        self.encoder = nn.Sequential(
            
            StdConvBlock(3,   32),
            StdConvBlock(32,  32),
            nn.MaxPool2d(2),            # 224 → 112

            StdConvBlock(32,  64),
            StdConvBlock(64,  64),
            nn.MaxPool2d(2),            # 112 → 56

            StdConvBlock(64,  128),
            StdConvBlock(128, 128),
            nn.MaxPool2d(2),            # 56  → 28

            StdConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))  # → 1×1
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.head(self.encoder(x))



In [12]:
class DSCNN(nn.Module):


    def __init__(self, num_classes=3):
        super().__init__()
        self.encoder = nn.Sequential(
        
            DSConvBlock(3,   32),
            DSConvBlock(32,  32),
            nn.MaxPool2d(2),

            DSConvBlock(32,  64),
            DSConvBlock(64,  64),
            nn.MaxPool2d(2),

            DSConvBlock(64,  128),
            DSConvBlock(128, 128),
            nn.MaxPool2d(2),

            DSConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.head(self.encoder(x))



def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"PlainCNN parameters : {count_params(PlainCNN()):,}")
print(f"DSCNN parameters    : {count_params(DSCNN()):,}")


PlainCNN parameters : 616,163
DSCNN parameters    : 104,260


In [13]:
class FocalLoss(nn.Module):


    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma
        self._ce  = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, targets):
        ce   = self._ce(logits, targets)
        p_t  = torch.exp(-ce)
        loss = ((1 - p_t) ** self.gamma) * ce
        return loss.mean()


## 5. Training Infrastructure

In [14]:
def run_training(
    model,
    train_loader,
    val_loader,
    num_epochs: int = 25,
    learning_rate: float = 1e-3,
    loss_fn=None,
):
   
    model = model.to(DEVICE)
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_val_acc  = 0.0
    best_weights  = copy.deepcopy(model.state_dict())

    for epoch in range(1, num_epochs + 1):
        
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for imgs, targets in train_loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            preds = model(imgs)
            loss  = loss_fn(preds, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct    += (preds.argmax(1) == targets).sum().item()
            total      += targets.size(0)

        train_acc  = correct / total
        avg_loss   = total_loss / len(train_loader)

        
        model.eval()
        val_correct, val_total = 0, 0

        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
                preds = model(imgs)
                val_correct += (preds.argmax(1) == targets).sum().item()
                val_total   += targets.size(0)

        val_acc = val_correct / val_total
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

        print(
            f"  Epoch {epoch:02d}/{num_epochs}  "
            f"loss={avg_loss:.4f}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_weights)
    print(f"  → Best val acc: {best_val_acc:.4f}")
    return model


In [15]:
CLASS_NAMES = ["Normal", "Benign", "Malignant"]

def evaluate(model, loader, label="Model"):
   
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(DEVICE)
            y_pred.extend(model(imgs).argmax(1).cpu().tolist())
            y_true.extend(targets.tolist())

    print(f"\n{'='*50}")
    print(f" {label}")
    print('='*50)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    return y_true, y_pred


## 6. Experiments

### Experiment 1 — Plain CNN (Baseline)

In [17]:
torch.cuda.empty_cache()

In [18]:
print("Training Plain CNN (Baseline) ...")
plain_model = run_training(
    PlainCNN(num_classes=3),
    train_loader,
    val_loader,
    num_epochs=25,
    learning_rate=1e-3,
    loss_fn=nn.CrossEntropyLoss()
)

plain_true, plain_pred = evaluate(plain_model, test_loader, "Plain CNN — Baseline")


Training Plain CNN (Baseline) ...
  Epoch 01/25  loss=0.9560  train_acc=0.5666  val_acc=0.3000
  Epoch 02/25  loss=0.9220  train_acc=0.5933  val_acc=0.2900
  Epoch 03/25  loss=0.8436  train_acc=0.6377  val_acc=0.6100
  Epoch 04/25  loss=0.7933  train_acc=0.6448  val_acc=0.6600
  Epoch 05/25  loss=0.7821  train_acc=0.6501  val_acc=0.5700
  Epoch 06/25  loss=0.7122  train_acc=0.6980  val_acc=0.6400
  Epoch 07/25  loss=0.6980  train_acc=0.7176  val_acc=0.6200
  Epoch 08/25  loss=0.6647  train_acc=0.7265  val_acc=0.7000
  Epoch 09/25  loss=0.6088  train_acc=0.7478  val_acc=0.6700
  Epoch 10/25  loss=0.6193  train_acc=0.7584  val_acc=0.5900
  Epoch 11/25  loss=0.6377  train_acc=0.7460  val_acc=0.6300
  Epoch 12/25  loss=0.5691  train_acc=0.7762  val_acc=0.7000
  Epoch 13/25  loss=0.5461  train_acc=0.7869  val_acc=0.6900
  Epoch 14/25  loss=0.5442  train_acc=0.7744  val_acc=0.7700
  Epoch 15/25  loss=0.5308  train_acc=0.7975  val_acc=0.7800
  Epoch 16/25  loss=0.4843  train_acc=0.8206  val_a

### Experiment 2 — Depthwise-Separable CNN (vanilla)

In [19]:
print("Training Depthwise-Separable CNN ...")
ds_model = run_training(
    DSCNN(num_classes=3),
    train_loader,
    val_loader,
    num_epochs=25,
    learning_rate=1e-3,
    loss_fn=nn.CrossEntropyLoss()
)

ds_true, ds_pred = evaluate(ds_model, test_loader, "Depthwise-Separable CNN (vanilla)")


Training Depthwise-Separable CNN ...
  Epoch 01/25  loss=0.9730  train_acc=0.5560  val_acc=0.2700
  Epoch 02/25  loss=0.9037  train_acc=0.5915  val_acc=0.2700
  Epoch 03/25  loss=0.8664  train_acc=0.6270  val_acc=0.4100
  Epoch 04/25  loss=0.8089  train_acc=0.6394  val_acc=0.6200
  Epoch 05/25  loss=0.7486  train_acc=0.6856  val_acc=0.6500
  Epoch 06/25  loss=0.7021  train_acc=0.7087  val_acc=0.6200
  Epoch 07/25  loss=0.6402  train_acc=0.7531  val_acc=0.6600
  Epoch 08/25  loss=0.6090  train_acc=0.7691  val_acc=0.5200
  Epoch 09/25  loss=0.5551  train_acc=0.7815  val_acc=0.6400
  Epoch 10/25  loss=0.4841  train_acc=0.8099  val_acc=0.7200
  Epoch 11/25  loss=0.4134  train_acc=0.8472  val_acc=0.5300
  Epoch 12/25  loss=0.4355  train_acc=0.8295  val_acc=0.7300
  Epoch 13/25  loss=0.4281  train_acc=0.8597  val_acc=0.6200
  Epoch 14/25  loss=0.4043  train_acc=0.8508  val_acc=0.7000
  Epoch 15/25  loss=0.3336  train_acc=0.8810  val_acc=0.8000
  Epoch 16/25  loss=0.2611  train_acc=0.9076  va

### Experiment 3 — DS-CNN + Weighted Random Oversampling

In [20]:
print("Training DS-CNN with Weighted Oversampling ...")
ds_over_model = run_training(
    DSCNN(num_classes=3),
    train_loader_over,          
    val_loader,
    num_epochs=25,
    learning_rate=1e-3,
    loss_fn=nn.CrossEntropyLoss()
)

ds_over_true, ds_over_pred = evaluate(ds_over_model, test_loader, "DS-CNN + Weighted Oversampling")


Training DS-CNN with Weighted Oversampling ...
  Epoch 01/25  loss=1.0638  train_acc=0.3517  val_acc=0.2700
  Epoch 02/25  loss=0.9878  train_acc=0.5187  val_acc=0.2700
  Epoch 03/25  loss=0.8923  train_acc=0.6306  val_acc=0.3300
  Epoch 04/25  loss=0.8371  train_acc=0.6199  val_acc=0.6300
  Epoch 05/25  loss=0.7652  train_acc=0.6856  val_acc=0.5700
  Epoch 06/25  loss=0.7097  train_acc=0.7016  val_acc=0.5500
  Epoch 07/25  loss=0.6661  train_acc=0.7140  val_acc=0.5500
  Epoch 08/25  loss=0.5967  train_acc=0.7549  val_acc=0.4700
  Epoch 09/25  loss=0.4887  train_acc=0.8117  val_acc=0.6800
  Epoch 10/25  loss=0.4458  train_acc=0.8259  val_acc=0.6100
  Epoch 11/25  loss=0.3697  train_acc=0.8703  val_acc=0.7100
  Epoch 12/25  loss=0.3352  train_acc=0.8757  val_acc=0.5400
  Epoch 13/25  loss=0.3313  train_acc=0.8810  val_acc=0.7700
  Epoch 14/25  loss=0.2935  train_acc=0.8934  val_acc=0.7000
  Epoch 15/25  loss=0.2994  train_acc=0.8917  val_acc=0.5300
  Epoch 16/25  loss=0.2120  train_acc=

### Experiment 4 — DS-CNN + Data Augmentation

In [21]:
print("Training DS-CNN with Data Augmentation ...")
ds_aug_model = run_training(
    DSCNN(num_classes=3),
    train_loader_aug,            
    val_loader,
    num_epochs=25,
    learning_rate=1e-3,
    loss_fn=nn.CrossEntropyLoss()
)

ds_aug_true, ds_aug_pred = evaluate(ds_aug_model, test_loader, "DS-CNN + Data Augmentation")


Training DS-CNN with Data Augmentation ...
  Epoch 01/25  loss=0.9788  train_acc=0.5471  val_acc=0.2700
  Epoch 02/25  loss=0.9375  train_acc=0.5773  val_acc=0.2700
  Epoch 03/25  loss=0.9256  train_acc=0.6075  val_acc=0.5600
  Epoch 04/25  loss=0.8923  train_acc=0.6092  val_acc=0.5800
  Epoch 05/25  loss=0.8404  train_acc=0.6270  val_acc=0.6200
  Epoch 06/25  loss=0.8106  train_acc=0.6448  val_acc=0.6500
  Epoch 07/25  loss=0.7858  train_acc=0.6643  val_acc=0.6700
  Epoch 08/25  loss=0.7534  train_acc=0.6554  val_acc=0.6400
  Epoch 09/25  loss=0.7043  train_acc=0.6927  val_acc=0.6600
  Epoch 10/25  loss=0.6840  train_acc=0.6856  val_acc=0.6600
  Epoch 11/25  loss=0.6743  train_acc=0.7123  val_acc=0.6300
  Epoch 12/25  loss=0.6548  train_acc=0.7425  val_acc=0.7200
  Epoch 13/25  loss=0.6036  train_acc=0.7584  val_acc=0.6900
  Epoch 14/25  loss=0.6047  train_acc=0.7389  val_acc=0.7500
  Epoch 15/25  loss=0.5871  train_acc=0.7780  val_acc=0.8000
  Epoch 16/25  loss=0.6031  train_acc=0.74

### Experiment 5 — DS-CNN + Focal Loss

In [22]:
print("Training DS-CNN with Focal Loss ...")
ds_focal_model = run_training(
    DSCNN(num_classes=3),
    train_loader,
    val_loader,
    num_epochs=25,
    learning_rate=1e-3,
    loss_fn=FocalLoss(gamma=2.0)  
)

ds_focal_true, ds_focal_pred = evaluate(ds_focal_model, test_loader, "DS-CNN + Focal Loss")


Training DS-CNN with Focal Loss ...
  Epoch 01/25  loss=0.4221  train_acc=0.5755  val_acc=0.2700
  Epoch 02/25  loss=0.4002  train_acc=0.5844  val_acc=0.2700
  Epoch 03/25  loss=0.3875  train_acc=0.5861  val_acc=0.3400
  Epoch 04/25  loss=0.3431  train_acc=0.6288  val_acc=0.6100
  Epoch 05/25  loss=0.2977  train_acc=0.6554  val_acc=0.6800
  Epoch 06/25  loss=0.2819  train_acc=0.6856  val_acc=0.6900
  Epoch 07/25  loss=0.2327  train_acc=0.7211  val_acc=0.6700
  Epoch 08/25  loss=0.2218  train_acc=0.7496  val_acc=0.6300
  Epoch 09/25  loss=0.1978  train_acc=0.7655  val_acc=0.6700
  Epoch 10/25  loss=0.1596  train_acc=0.8046  val_acc=0.6300
  Epoch 11/25  loss=0.1298  train_acc=0.8313  val_acc=0.7200
  Epoch 12/25  loss=0.1198  train_acc=0.8472  val_acc=0.6500
  Epoch 13/25  loss=0.1160  train_acc=0.8526  val_acc=0.6100
  Epoch 14/25  loss=0.0933  train_acc=0.8917  val_acc=0.3800
  Epoch 15/25  loss=0.0780  train_acc=0.8988  val_acc=0.6300
  Epoch 16/25  loss=0.0677  train_acc=0.9361  val

## 7. Final Comparison

In [23]:
experiments = [
    ("Plain CNN (Baseline)",            plain_true,    plain_pred),
    ("DS-CNN (Vanilla)",                ds_true,       ds_pred),
    ("DS-CNN + Oversampling",           ds_over_true,  ds_over_pred),
    ("DS-CNN + Augmentation",           ds_aug_true,   ds_aug_pred),
    ("DS-CNN + Focal Loss",             ds_focal_true, ds_focal_pred),
]

summary = pd.DataFrame({
    "Experiment": [e[0] for e in experiments],
    "Test Accuracy": [round(accuracy_score(e[1], e[2]), 4) for e in experiments],
    "Weighted F1":   [round(f1_score(e[1], e[2], average="weighted"), 4) for e in experiments],
})

summary_sorted = summary.sort_values("Weighted F1", ascending=False).reset_index(drop=True)
summary_sorted.index += 1   # 1-based rank

print("\n" + "="*60)
print("       FINAL MODEL COMPARISON — BUSI Dataset")
print("="*60)
print(summary_sorted.to_string())
print("="*60)



       FINAL MODEL COMPARISON — BUSI Dataset
              Experiment  Test Accuracy  Weighted F1
1   Plain CNN (Baseline)         0.8034       0.8002
2  DS-CNN + Oversampling         0.7863       0.7848
3  DS-CNN + Augmentation         0.7778       0.7818
4       DS-CNN (Vanilla)         0.7863       0.7734
5    DS-CNN + Focal Loss         0.7521       0.7429
